# W12 — Full Pipeline with Extended BI (18 variables) — Stages 1 & 2

**Purpose.** Re-run the article's experiments with the extended BI set and answer the advisor's evaluation points from the same trained models, so that no experiment is run twice.

**Extended BI set (18 variables).**

| Group | Variables | Type |
|---|---|---|
| Operational (13) | pm/cm cost, labor rates (2), production priority, downtime penalty, revenue per hour, spare parts available, lead time, technician available, maintenance window, shift, contract penalty | Time-varying (Δ = 10/25/50 cycles); no lifetime information |
| Asset history (5) | component batch, supplier, last overhaul type, prior unplanned removals, hours since overhaul at entry | Static per engine; controlled link to lifetime |

**Link strength ρ = 0.4, fixed before running.** ρ = 0.4 is the strongest level that passed every realism check in all 5 simulated fleets of W11 (ρ = 0.6 failed one check in one fleet). It is chosen from realism, not from RMSE. Fleet 2026 (the generator files in the repo) is the reference fleet.

**Protocol.** Asset history must be consistent with each engine's full life, which for the official test engines is only known through the true RUL. All experiments therefore use the run-to-failure training engines of C-MAPSS (Saxena et al., 2008): **FD001 + FD003 analysed as one fleet of 200 engines** (`DATASETS` in Section 0; FD003 unit ids offset by +1000; `['FD001']` reproduces the 100-engine runs). FD003 has the same single operating condition as FD001 and two fault modes. Operational BI uses one cost profile for the whole fleet, so it cannot identify the subset. Protocol:
- 5-fold cross-validation **by engine**;
- inside each training fold, 20% of engines held out as an **inner validation set** (early stopping of the BiLSTM and fusion weight α), so the test fold is never used for any choice;
- feature selection and scaling **fitted inside each fold**, on training engines only;
- each test engine truncated at **3 random points** (10–95% of its life), as NASA built the official test set; RMSE is computed on the RUL at the truncation point.

> **Why truncation.** Scoring the last window of a run-to-failure engine always means predicting RUL ≈ 0, which is trivially easy. That is why earlier CV RMSEs (≈ 2.4–3.9 in W05/W07) were far below test RMSEs (≈ 13–14). Truncation makes CV comparable to the official test protocol.

**Models.** Hyperparameters are selected **per feature configuration** by W13 (grid search, grouped K-fold with truncation) and read from Drive: **run W13 first**.
- ML branch: XGBoost, correlation-based feature selection, window representation (statistics or flatten) chosen by W13. Random Forest (300 trees, depth 15) on the same representation is the tree-based benchmark.
- DL branch: BiLSTM with dual attention, stratified AFICv selection (90% cumulative importance).
- Hybrid: (i) α-weighted fusion, α = inverse-RMSE weight on the inner validation set; (ii) learned dense fusion, a small network trained on the inner-validation predictions [ŷ_DL, ŷ_ML, |ŷ_DL − ŷ_ML|].
- **Routing of asset history: ML branch only.** A first run gave the static asset-history variables to both branches (as constant input channels for the BiLSTM). The BiLSTM then overfitted: the 12 constant columns act as an engine fingerprint, so it memorised training engines (training error down, validation error up, early stopping after ~26 epochs instead of ~48; DL RMSE 18.1 → 24.8). Static per-engine records are therefore given to the tree-based branch only, while the BiLSTM keeps the time-varying inputs (sensors and operational BI). Section 4.1 reproduces this comparison from the cache.

**Stages in this notebook.**

| Stage | Trains models? | Advisor points |
|---|---|---|
| 1. Main experiment: 3 feature sets × (XGBoost, RF, BiLSTM-DA, α fusion, dense fusion) × 3 seeds, plus 2 shuffled-BI controls on the same models | Yes (cached) | Article tables (baselines, fusion), BI falsification with the final hybrid model, **4** repeated CV |
| 2. Analyses from cached predictions | No | **5** statistics, **6** complementarity, **10/21** conformal intervals and failure risk, **15** explanation stability, **16** permutation importance, **18** subgroups, **19** late vs early errors, **22** computing time |

Stages 3–5 (points 8, 9, 13, 14) are separate notebooks that reuse the same functions and cache.

## 0. Setup

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from datetime import datetime

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    # Fresh clone every run; cache and results live on Drive so finished work survives a disconnect
    if os.path.exists('/content/PhD-Project-'):
        !rm -rf /content/PhD-Project-
    !git clone https://github.com/f-khadija-benzine/PhD-Project-.git /content/PhD-Project-
    !pip install xgboost shap -q
    os.chdir('/content/PhD-Project-/src')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/PhD-Project-')
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
    SAVE_DIR = Path(f'/content/drive/MyDrive/PhD_results/W12_{TIMESTAMP}')
    CACHE_ROOT = Path('/content/drive/MyDrive/PhD_results/W12_cache')
else:
    PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    SAVE_DIR = None
    CACHE_ROOT = PROJECT_ROOT / 'results' / 'W12_cache'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

CMAPSS_DIR = PROJECT_ROOT / 'data' / 'raw_data' / 'C_MAPSS'
BI_DIR     = PROJECT_ROOT / 'data' / 'raw_data' / 'BI_data'
ASSET_DIR  = BI_DIR / 'asset'
# ---- Fleet: FD001 alone, or FD001 + FD003 analysed as ONE fleet of 200 engines ----
# FD003: same single operating condition as FD001, two fault modes (HPC + fan).
# Unit ids of FD003 are offset by +1000. Set DATASETS = ['FD001'] to reproduce the FD001-only runs.
DATASETS = ['FD001', 'FD003']
for _ds in DATASETS:
    assert (CMAPSS_DIR / f'train_{_ds}.txt').exists(), f"train_{_ds}.txt missing in {CMAPSS_DIR}: upload it to the repo"
FLEET_TAG = '_'.join(DATASETS)
TAG_SUFFIX = '' if FLEET_TAG == 'FD001' else f'_{FLEET_TAG}'   # FD001 keeps its original paths and caches

RESULTS    = PROJECT_ROOT / 'results' / f'W12{TAG_SUFFIX}'
CACHE_ROOT = CACHE_ROOT.parent / f'{CACHE_ROOT.name}{TAG_SUFFIX}'
if SAVE_DIR is not None:
    SAVE_DIR = SAVE_DIR.parent / f'{SAVE_DIR.name}{TAG_SUFFIX}'
for d in (RESULTS, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)
if SAVE_DIR is not None:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Fleet: {FLEET_TAG}\nProject root: {PROJECT_ROOT}\nResults: {RESULTS}\nCache: {CACHE_ROOT}\nDrive save: {SAVE_DIR}")

In [ ]:
# Regenerate both BI blocks from the generators in the repo, so data and code always match
for script in ['generate_bi_data.py', 'generate_bi_asset_data.py']:
    res = subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / script)],
                         capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    print(f"--- {script}: {'OK' if res.returncode == 0 else 'FAILED'}")
    if res.returncode != 0:
        print(res.stderr[-2000:])
        raise RuntimeError(f"{script} failed")

In [ ]:
import importlib, time, json, io, contextlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import shap

import ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, generate_bi_asset_data as gba
for m in (ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, gba):
    importlib.reload(m)
from ml_branch import MLBranch
from attention import build_dual_attention_bilstm
from feature_selection import BIAwareFeatureSelector
from feature_selection_aficv import AFICvFeatureSelector
from bi_fusion import CATEGORICAL_BI_VARS, CONTINUOUS_BI_VARS, BI_FEATURE_NAMES

# ---------------- Experiment configuration (fixed before running) ----------------
RHO = 0.4                      # asset-history link strength, chosen from W11 realism checks
FLEET = 2026                   # reference simulated fleet (= repo files)
SEEDS = [42, 43, 44]           # repeated CV (advisor point 4)
W = 30
RUL_MAX = 125
N_FOLDS = 5
N_CUTS = 3
INNER_VAL_FRAC = 0.2
CUT_SEED = 42                  # same truncation points as W11
ASSET_ROUTING = 'ml_only'      # static asset history goes to the ML branch only (see Section 4.1)

RF_PARAMS = dict(n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2)   # benchmark, not tuned
BATCH, EPOCHS = 128, 100
DENSE_EPOCHS = 300           # dense fusion network (small, early-stopped; independent of QUICK)
CACHE_VERSION = 'v2'           # v2 = hyperparameters from W13 (v1 runs used the W05/W07 settings)
CORR_SEL = dict(variance_threshold=0.01, correlation_threshold=0.95)
AFICV_SEL = dict(base_learner='xgboost', n_folds=5, cumulative_threshold=0.90)

# QUICK = True: smoke test (tiny models, 1 seed). Use False for reported results.
QUICK = False
if QUICK:
    SEEDS = [42]; EPOCHS = 2
    RF_PARAMS = dict(n_estimators=30, max_depth=8, min_samples_split=5, min_samples_leaf=2)

GPU = tf.config.list_physical_devices('GPU')
XGB_EXTRA = {'device': 'cuda', 'tree_method': 'hist'} if GPU else {}
CACHE_DIR = CACHE_ROOT / ('quick' if QUICK else 'full')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"QUICK={QUICK} | seeds={SEEDS} | epochs={EPOCHS} | GPU={bool(GPU)} | cache={CACHE_DIR}")

## 1. Data: sensors + 18 BI variables (training engines only)

In [ ]:
cols = ['unit', 'cycle'] + [f'setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
def _load_train(ds):
    t = pd.read_csv(CMAPSS_DIR / f'train_{ds}.txt', sep=r'\s+', header=None, names=cols, engine='python')
    t['unit'] += gba.UNIT_OFFSET[ds]
    t['subset'] = ds
    return t
train = pd.concat([_load_train(ds) for ds in DATASETS], ignore_index=True)
train['total_life'] = train.groupby('unit')['cycle'].transform('max')
train['rul'] = (train['total_life'] - train['cycle']).clip(upper=RUL_MAX)
UNITS = np.sort(train['unit'].unique())
LIFE = train.groupby('unit')['total_life'].first()

SENSOR_COLS = [c for c in train.columns if c.startswith('sensor_')]
SETTING_COLS = [c for c in train.columns if c.startswith('setting_')]

# Operational BI (13 variables)
bi_op = pd.read_csv(BI_DIR / f'BI_{FLEET_TAG}_train.csv').rename(columns={'unit_id': 'unit'})
OP_VARS = [c for c in BI_FEATURE_NAMES if c in bi_op.columns]
OP_CAT = [c for c in CATEGORICAL_BI_VARS if c in OP_VARS]
OP_CONT = [c for c in CONTINUOUS_BI_VARS if c in OP_VARS]

# Asset-history BI (5 variables), reference fleet, rho fixed above
asset = pd.read_csv(ASSET_DIR / f"BI_{FLEET_TAG}_asset_{gba.rho_tag(RHO)}.csv").rename(columns={'unit_id': 'unit'})
ASSET_VARS = gba.ASSET_VARS
ASSET_CONT = ['prior_unplanned_removals', 'hours_since_overhaul_entry']
BATCH_LEVELS = [f"B{i}" for i in range(1, gba.N_BATCHES + 1)]

def build_table(op_df, asset_df):
    # One table with sensors + all BI, categorical variables one-hot encoded
    df = train[['unit', 'cycle', 'rul'] + SETTING_COLS + SENSOR_COLS].merge(
        op_df[['unit', 'cycle'] + OP_VARS], on=['unit', 'cycle'], how='left')
    for c in OP_CAT:
        df[c] = pd.Categorical(df[c].astype(int), categories=sorted(bi_op[c].unique()))
    df = pd.get_dummies(df, columns=OP_CAT, dtype=float)
    a = asset_df[['unit'] + ASSET_VARS].copy()
    a['component_batch'] = pd.Categorical(a['component_batch'], categories=BATCH_LEVELS)
    df = df.merge(pd.get_dummies(a, columns=['component_batch'], dtype=float), on='unit', how='left')
    return df.sort_values(['unit', 'cycle']).reset_index(drop=True)

# Falsification controls (same methods as W10/W11):
#  - operational BI: each variable permuted independently across all rows
#  - asset history: full records permuted across engines (stays constant within an engine)
op_shuf = bi_op.copy()
r_op = np.random.RandomState(42)
for c in OP_VARS:
    op_shuf[c] = r_op.permutation(op_shuf[c].values)
asset_shuf = asset.sort_values('unit').reset_index(drop=True).copy()
perm = np.random.default_rng(FLEET + 1000).permutation(len(asset_shuf))
asset_shuf[ASSET_VARS] = asset_shuf[ASSET_VARS].values[perm]

DATA = {'real': build_table(bi_op, asset),
        'asset_shuffled': build_table(bi_op, asset_shuf),
        'all_shuffled': build_table(op_shuf, asset_shuf)}
DF = DATA['real']
assert all(list(d.columns) == list(DF.columns) for d in DATA.values())

OP_COLS = [c for c in DF.columns if c in OP_CONT or c in OP_VARS or c.startswith(tuple(f"{v}_" for v in OP_CAT))]
ASSET_COLS = [c for c in DF.columns if c.startswith(tuple(ASSET_VARS))]
FEATURE_SETS = {
    'S':        {'bi': [],                    'data': 'real'},            # sensors + settings only
    'S+Op':     {'bi': OP_COLS,               'data': 'real'},            # + 13 operational BI (article's original BI)
    'S+Op+A':   {'bi': OP_COLS + ASSET_COLS,  'data': 'real'},            # + 5 asset-history BI = 18 BI variables
    'S+Op+A~':  {'bi': OP_COLS + ASSET_COLS,  'data': 'asset_shuffled'},  # control: asset history shuffled across engines
    'S+Op~+A~': {'bi': OP_COLS + ASSET_COLS,  'data': 'all_shuffled'},    # control: all 18 BI shuffled
}
MAIN_FSETS = ['S', 'S+Op', 'S+Op+A']
CONT_COLS = SENSOR_COLS + SETTING_COLS + OP_CONT + ASSET_CONT
GROUP_OF = {**{c: 'sensor' for c in SENSOR_COLS + SETTING_COLS},
            **{c: 'operational BI' for c in OP_COLS}, **{c: 'asset BI' for c in ASSET_COLS}}

print(f"Fleet {FLEET_TAG}: " + ", ".join(f"{s} {n} engines" for s, n in train.groupby("subset").unit.nunique().items()))
print(f"Engines: {len(UNITS)}, rows: {len(DF)}, life [{LIFE.min()}, {LIFE.max()}]")
print(f"Columns: {len(SENSOR_COLS)} sensors + {len(SETTING_COLS)} settings + {len(OP_COLS)} operational BI "
      f"(13 variables) + {len(ASSET_COLS)} asset BI (5 variables)")

### Hyperparameters selected by W13

In [ ]:
BEST_PARAMS_PATH = (Path(f'/content/drive/MyDrive/PhD_results/W13_best_params{TAG_SUFFIX}.json') if IN_COLAB
                    else PROJECT_ROOT / 'results' / f'W13{TAG_SUFFIX}' / f'W13_best_params{TAG_SUFFIX}.json')
DL_FIXED = {'feature_attention_dim': 32, 'dropout_rate': 0.3}
if QUICK:
    _ml = {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.1, 'flatten': 'statistics'}
    _dl = {'lstm_units': 32, 'temporal_attention_dim': 64, 'dense_units': 32, 'learning_rate': 0.001, **DL_FIXED}
    TUNED = {f: {'ml': _ml, 'dl': _dl} for f in MAIN_FSETS}
else:
    if not BEST_PARAMS_PATH.exists():
        raise FileNotFoundError(f"{BEST_PARAMS_PATH} not found: run W13 (hyperparameter optimisation) first.")
    _b = json.loads(BEST_PARAMS_PATH.read_text())
    if _b.get('quick'):
        raise ValueError("W13 was run with QUICK = True: re-run W13 with QUICK = False.")
    TUNED = _b['params']

def params_for(fset):
    # Shuffled controls use the settings of the 18-BI configuration they are compared with
    return TUNED[fset if fset in TUNED else 'S+Op+A']

display(pd.DataFrame({f: {**{f"ML {k}": v for k, v in TUNED[f]['ml'].items()},
                          **{f"DL {k}": v for k, v in TUNED[f]['dl'].items()}} for f in MAIN_FSETS}))

## 2. Folds, inner validation and truncation points

Fixed once, shared by every model, feature set and seed (paired comparisons).

In [ ]:
folds = list(GroupKFold(n_splits=N_FOLDS).split(UNITS, groups=UNITS))

rng = np.random.default_rng(CUT_SEED)
cut_rows = []
for u in UNITS:
    life = int(LIFE[u])
    lo, hi = max(int(0.10 * life), 1), max(int(0.95 * life), 2)
    for k, c in enumerate(np.sort(rng.integers(lo, hi, N_CUTS))):
        cut_rows.append({'unit': u, 'cut_id': k, 'cut_cycle': int(c), 'true_rul': min(life - int(c), RUL_MAX)})
CUTS = pd.DataFrame(cut_rows)

def inner_split(train_units, seed):
    r = np.random.default_rng(seed)
    val = r.choice(train_units, size=int(round(INNER_VAL_FRAC * len(train_units))), replace=False)
    return np.setdiff1d(train_units, val), np.sort(val)

assert all(len(set(UNITS[tr]) & set(UNITS[te])) == 0 for tr, te in folds)
print(f"{N_FOLDS} folds × {[len(te) for _, te in folds]} test engines; "
      f"{len(CUTS)} truncation points, true RUL {CUTS.true_rul.min()}–{CUTS.true_rul.max()}")

## 3. Pipeline functions

- **Scaling and feature selection** are fitted on the training engines of each fold only.
- **Static asset features:** the ML branch receives them **once** per sample after flattening (repeating a constant inside the window would create 6 identical statistics). With `ASSET_ROUTING = 'ml_only'` the DL branch does not receive them; with `'both'` (first run, kept for comparison) it receives them as constant input channels.
- **Explainability per fold:** SHAP (ML), permutation importance (ML) and mean feature-attention weight (DL), all computed on the held-out truncation points. They are stored with the predictions, so points 15 and 16 need no retraining.

In [ ]:
STAT_SUFFIX = ('_mean', '_std', '_min', '_max', '_last', '_trend')
def is_static(c): return c.startswith(tuple(ASSET_VARS))
def original_name(flat):
    for s in STAT_SUFFIX:
        if flat.endswith(s):
            return flat[: -len(s)]
    return flat

def quiet(fn, *args, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kw)

def scale_fold(train_units, fset):
    d = DATA[FEATURE_SETS[fset]['data']].copy()
    sc = MinMaxScaler().fit(d.loc[d.unit.isin(train_units), CONT_COLS])
    d[CONT_COLS] = sc.transform(d[CONT_COLS])
    return d

def select_features(d, train_units, fset, method, exclude_asset=False):
    # Returns the selected column list, fitted on training engines only
    tr = d[d.unit.isin(train_units)]
    bi = FEATURE_SETS[fset]['bi']
    if exclude_asset:
        bi = [c for c in bi if c not in ASSET_COLS]
    if method == 'correlation':
        sel = BIAwareFeatureSelector(**CORR_SEL)
        return quiet(sel.select_features, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                     setting_cols=SETTING_COLS, exclude_cols=['unit', 'cycle', 'rul'])
    if method == 'aficv':
        sel = AFICvFeatureSelector(**AFICV_SEL)
        if bi:
            return quiet(sel.select_features_stratified, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                         setting_cols=SETTING_COLS, target_col='rul', group_col='unit')
        return quiet(sel.select_features, data=tr, feature_cols=SENSOR_COLS + SETTING_COLS,
                     target_col='rul', group_col='unit')
    raise ValueError(method)

def windows_train(d, units, feats):
    # PAD=False as in the article: windows ending at cycles W..T of each engine
    X, y, owner = [], [], []
    for u in units:
        g = d[d.unit == u]
        v = g[feats].to_numpy(np.float32)
        if len(v) < W:
            continue
        X.append(sliding_window_view(v, (W, v.shape[1]))[:, 0])
        y.append(g['rul'].to_numpy(np.float32)[W - 1:])
        owner += [u] * (len(v) - W + 1)
    return np.concatenate(X), np.concatenate(y), np.asarray(owner)

def windows_cuts(d, units, feats):
    # Last window before each truncation point (zero-padded if the engine is shorter than W)
    X, keys = [], []
    for u in units:
        v_full = d[d.unit == u][feats].to_numpy(np.float32)
        for _, c in CUTS[CUTS.unit == u].iterrows():
            v = v_full[:c.cut_cycle]
            if len(v) < W:
                v = np.vstack([np.zeros((W - len(v), len(feats)), np.float32), v])
            X.append(v[-W:]); keys.append((u, int(c.cut_id)))
    return np.asarray(X), keys

def static_rows(d, owners, static_cols):
    return d.groupby('unit')[static_cols].first().loc[list(owners)].to_numpy(np.float32)

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

In [ ]:
def train_ml(d, fit_units, val_units, test_units, feats, seed, p):
    # XGBoost (tuned) and Random Forest benchmark on the same window representation
    dyn = [c for c in feats if not is_static(c)]
    stat = [c for c in feats if is_static(c)]
    Xtr, ytr, otr = windows_train(d, fit_units, dyn)
    Xva, yva, ova = windows_train(d, val_units, dyn)
    Xte, keys = windows_cuts(d, test_units, dyn)
    ml = MLBranch(model_type='xgboost', flatten_strategy=p['flatten'], random_state=seed,
                  n_estimators=p['n_estimators'], max_depth=p['max_depth'], learning_rate=p['learning_rate'], **XGB_EXTRA)
    Ftr, Fva, Fte = ml._flatten(Xtr, dyn), ml._flatten(Xva, dyn), ml._flatten(Xte, dyn)
    names = list(ml.feature_names_flat)
    if stat:
        Ftr = np.hstack([Ftr, static_rows(d, otr, stat)])
        Fva = np.hstack([Fva, static_rows(d, ova, stat)])
        Fte = np.hstack([Fte, static_rows(d, [u for u, _ in keys], stat)])
        names += stat
    t0 = time.perf_counter(); ml.model.fit(Ftr, ytr); t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); p_te = ml.model.predict(Fte); t_inf = (time.perf_counter() - t0) / len(Fte)
    p_va = ml.model.predict(Fva)
    y_te = CUTS.set_index(['unit', 'cut_id']).loc[keys, 'true_rul'].to_numpy(float)

    rf = RandomForestRegressor(random_state=seed, n_jobs=-1, **RF_PARAMS)
    t0 = time.perf_counter(); rf.fit(Ftr, ytr); t_train_rf = time.perf_counter() - t0
    p_rf = rf.predict(Fte)

    # Explainability on held-out truncation points: SHAP and permutation importance, per original variable
    t0 = time.perf_counter()
    sv = np.abs(shap.TreeExplainer(ml.model).shap_values(Fte)).mean(0)
    t_shap = time.perf_counter() - t0
    orig = np.array([original_name(n) for n in names])
    shap_imp = pd.Series(sv, index=orig).groupby(level=0).sum()
    base = rmse(y_te, p_te); r = np.random.default_rng(seed); perm = {}
    for var in np.unique(orig):
        idx = np.where(orig == var)[0]; inc = []
        for _ in range(10):
            Fp = Fte.copy(); Fp[:, idx] = Fp[r.permutation(len(Fp))][:, idx]
            inc.append(rmse(y_te, ml.model.predict(Fp)) - base)
        perm[var] = np.mean(inc)
    imp = pd.DataFrame({'shap_ml': shap_imp, 'perm_ml': pd.Series(perm)})
    return dict(pred=p_te, pred_rf=p_rf, keys=keys, val_rmse=rmse(yva, p_va), p_va=p_va, y_va=yva,
                t_train=t_train, t_train_rf=t_train_rf, t_inf=t_inf, t_shap=t_shap, imp=imp, n_feat=len(feats))

def train_dl(d, fit_units, val_units, test_units, feats, seed, p):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    Xtr, ytr, _ = windows_train(d, fit_units, feats)
    Xva, yva, _ = windows_train(d, val_units, feats)
    Xte, keys = windows_cuts(d, test_units, feats)
    model, att = build_dual_attention_bilstm(window_size=W, n_features=len(feats), **p)
    cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
          tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
    t0 = time.perf_counter()
    hist = model.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH, validation_data=(Xva, yva), callbacks=cb, verbose=0)
    t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); p_te = model.predict(Xte, batch_size=256, verbose=0).ravel()
    t_inf = (time.perf_counter() - t0) / len(Xte)
    p_va = model.predict(Xva, batch_size=1024, verbose=0).ravel()
    fw = att.predict(Xte, batch_size=256, verbose=0)['feature_weights']      # (n, W, n_feat)
    imp = pd.DataFrame({'attn_dl': pd.Series(np.asarray(fw).mean(axis=(0, 1)), index=feats)})
    return dict(pred=p_te, keys=keys, val_rmse=rmse(yva, p_va), p_va=p_va, y_va=yva, t_train=t_train, t_inf=t_inf,
                imp=imp, n_feat=len(feats), epochs=len(hist.history['loss']))

def dense_fusion(ml, dl, seed):
    # Learned fusion: small network on [y_DL, y_ML, |y_DL - y_ML|], trained on inner-validation predictions only
    assert np.allclose(ml['y_va'], dl['y_va'])
    # Inputs and target scaled by RUL_MAX for stable training; output rescaled to cycles
    feats = lambda a, b: np.column_stack([a, b, np.abs(a - b)]) / RUL_MAX
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    inp = tf.keras.layers.Input(shape=(3,))
    x = tf.keras.layers.Dense(8, activation='relu')(inp)
    x = tf.keras.layers.Dropout(0.2)(x)
    net = tf.keras.Model(inp, tf.keras.layers.Dense(1)(x))
    net.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    net.fit(feats(dl['p_va'], ml['p_va']), ml['y_va'] / RUL_MAX, epochs=DENSE_EPOCHS, batch_size=64,
            validation_split=0.2, verbose=0,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)])
    return net.predict(feats(dl['pred'], ml['pred']), verbose=0).ravel() * RUL_MAX

In [ ]:
def run_fold(fset, seed, k, routing=None):
    # One (feature set, seed, fold): ML + DL + hybrid. Cached on disk; returns predictions, importances, meta.
    # routing: 'ml_only' -> asset history to the ML branch only; 'both' -> to both branches (first run).
    routing = routing or ASSET_ROUTING
    has_asset = any(c in ASSET_COLS for c in FEATURE_SETS[fset]['bi'])
    tag = f"{CACHE_VERSION}_{fset}_s{seed}_f{k}".replace('+', 'p').replace('~', 'sh')
    if has_asset and routing == 'ml_only':
        tag += '_mlonly'
    p_pred, p_imp, p_meta = (CACHE_DIR / f"{tag}_{x}" for x in ('pred.csv', 'imp.csv', 'meta.json'))
    if p_pred.exists() and p_imp.exists() and p_meta.exists():
        p_, i_, m_ = pd.read_csv(p_pred), pd.read_csv(p_imp), json.loads(p_meta.read_text())
        p_['routing'] = routing if has_asset else 'n/a'; m_['routing'] = routing if has_asset else 'n/a'
        return p_, i_, m_

    tr_idx, te_idx = folds[k]
    tr_u, te_u = UNITS[tr_idx], UNITS[te_idx]
    fit_u, val_u = inner_split(tr_u, seed * 100 + k)
    d = scale_fold(tr_u, fset)
    t0 = time.perf_counter()
    f_ml = select_features(d, tr_u, fset, 'correlation')
    f_dl = select_features(d, tr_u, fset, 'aficv', exclude_asset=(has_asset and routing == 'ml_only'))
    t_sel = time.perf_counter() - t0

    hp = params_for(fset)
    ml = train_ml(d, fit_u, val_u, te_u, f_ml, seed, hp['ml'])
    dl = train_dl(d, fit_u, val_u, te_u, f_dl, seed, hp['dl'])
    assert ml['keys'] == dl['keys']
    alpha = (1 / dl['val_rmse']) / (1 / ml['val_rmse'] + 1 / dl['val_rmse'])   # DL weight, inner validation only

    pred = pd.DataFrame(ml['keys'], columns=['unit', 'cut_id'])
    pred['ml'], pred['rf'], pred['dl'] = ml['pred'], ml['pred_rf'], dl['pred']
    pred['hybrid'] = alpha * pred['dl'] + (1 - alpha) * pred['ml']
    pred['hybrid_dense'] = dense_fusion(ml, dl, seed)
    pred = pred.merge(CUTS, on=['unit', 'cut_id'])
    pred[['fset', 'seed', 'fold']] = fset, seed, k
    pred['routing'] = routing if has_asset else 'n/a'
    imp = ml['imp'].join(dl['imp'], how='outer').rename_axis('feature').reset_index()
    imp[['fset', 'seed', 'fold']] = fset, seed, k
    meta = dict(fset=fset, seed=seed, fold=k, routing=routing if has_asset else 'n/a', alpha=alpha, ml_val_rmse=ml['val_rmse'], dl_val_rmse=dl['val_rmse'],
                n_feat_ml=ml['n_feat'], n_feat_dl=dl['n_feat'], features_ml=f_ml, features_dl=f_dl,
                t_select=t_sel, t_train_ml=ml['t_train'], t_train_rf=ml['t_train_rf'], t_train_dl=dl['t_train'],
                t_inf_ml=ml['t_inf'], t_inf_dl=dl['t_inf'], t_shap=ml['t_shap'], dl_epochs=dl['epochs'])
    pred.to_csv(p_pred, index=False); imp.to_csv(p_imp, index=False); p_meta.write_text(json.dumps(meta))
    return pred, imp, meta
print("Pipeline ready ✓")

## 4. Stage 1 — Main experiment

5 feature sets × 5 folds × 3 seeds = 75 runs, each training XGBoost and the dual-attention BiLSTM:
- the 3 main feature sets (sensors; + 13 operational BI; + 18 BI);
- 2 falsification controls with the **final hybrid model**: 18 BI with asset history shuffled across engines (`S+Op+A~`), and 18 BI with everything shuffled (`S+Op~+A~`).

Use a **GPU runtime** in Colab. Finished runs are cached on Drive: if Colab disconnects, re-run the notebook and it resumes.

In [ ]:
t0 = time.time(); P, I, M = [], [], []
for seed in SEEDS:
    for fset in FEATURE_SETS:
        for k in range(N_FOLDS):
            p, i, m = run_fold(fset, seed, k)
            P.append(p); I.append(i); M.append(m)
        last = pd.concat(P[-N_FOLDS:])
        print(f"seed {seed} | {fset:<9} | XGB {rmse(last.true_rul, last.ml):5.2f}  RF {rmse(last.true_rul, last.rf):5.2f}  "
              f"DL {rmse(last.true_rul, last.dl):5.2f}  Hybrid-α {rmse(last.true_rul, last.hybrid):5.2f}  "
              f"Hybrid-dense {rmse(last.true_rul, last.hybrid_dense):5.2f} | {(time.time() - t0) / 60:.1f} min")
PRED = pd.concat(P, ignore_index=True)
IMP = pd.concat(I, ignore_index=True)
META = pd.DataFrame(M)
PRED.to_csv(RESULTS / 'predictions.csv', index=False)
IMP.to_csv(RESULTS / 'importances.csv', index=False)
META.drop(columns=['features_ml', 'features_dl']).to_csv(RESULTS / 'run_meta.csv', index=False)
print(f"\nStage 1 done: {len(META)} runs, {len(PRED)} predictions")

### 4.1 Design check — routing of static asset history

18-BI configuration, same folds, seeds and hyperparameters, with asset history given to **both** branches (constant input channels for the BiLSTM) versus to the **ML branch only** (main configuration). The BiLSTM's validation error, its number of training epochs and the test RMSE show the overfitting that motivated the routing.

In [ ]:
ASSET_FSETS = ['S+Op+A']
rows = []
for routing in ['both', 'ml_only']:
    for fset in ASSET_FSETS:
        P_r, M_r = [], []
        for seed in SEEDS:
            for k in range(N_FOLDS):
                p, _, m = run_fold(fset, seed, k, routing)
                P_r.append(p); M_r.append(m)
        P_r, M_r = pd.concat(P_r), pd.DataFrame(M_r)
        rows.append({'asset history to': 'both branches' if routing == 'both' else 'ML branch only', 'Features': fset,
                     'RMSE ML': rmse(P_r.true_rul, P_r.ml), 'RMSE DL': rmse(P_r.true_rul, P_r.dl),
                     'RMSE Hybrid': rmse(P_r.true_rul, P_r.hybrid), 'DL inner-val RMSE': M_r.dl_val_rmse.mean(),
                     'DL epochs': M_r.dl_epochs.mean(), 'α (DL weight)': M_r.alpha.mean()})
s_ref = PRED[PRED.fset == 'S']
print(f"Reference, sensors only: RMSE ML {rmse(s_ref.true_rul, s_ref.ml):.2f} | DL {rmse(s_ref.true_rul, s_ref.dl):.2f} | "
      f"Hybrid {rmse(s_ref.true_rul, s_ref.hybrid):.2f}")
ROUTING = pd.DataFrame(rows).round(2)
ROUTING.to_csv(RESULTS / 'routing_check.csv', index=False)
display(ROUTING)

## 5. Stage 2 — Analyses from the cached predictions

### 5.1 Main results and repeated CV (point 4)

RMSE, MAE and NASA score per model and feature set: mean ± SD over the 3 seeds.

In [ ]:
MODELS = ['ml', 'rf', 'dl', 'hybrid', 'hybrid_dense']
MODEL_NAME = {'ml': 'ML (XGBoost)', 'rf': 'ML (Random Forest)', 'dl': 'DL (BiLSTM-DA)',
              'hybrid': 'Hybrid (α fusion)', 'hybrid_dense': 'Hybrid (dense fusion)'}
FSETS = list(FEATURE_SETS)

def nasa(y, p):
    d = np.asarray(p) - np.asarray(y)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

rows = []
for fset, model in itertools.product(FSETS, MODELS):
    per_seed = [(rmse(g.true_rul, g[model]), float(np.mean(np.abs(g[model] - g.true_rul))), nasa(g.true_rul, g[model]) / N_CUTS)
                for _, g in PRED[PRED.fset == fset].groupby('seed')]
    a = np.array(per_seed)
    rows.append({'Features': fset, 'Model': MODEL_NAME[model],
                 'RMSE': f"{a[:, 0].mean():.2f} ± {a[:, 0].std(ddof=1) if len(a) > 1 else 0:.2f}",
                 'MAE': f"{a[:, 1].mean():.2f} ± {a[:, 1].std(ddof=1) if len(a) > 1 else 0:.2f}",
                 'NASA score (per 100 engines)': f"{a[:, 2].mean():.0f} ± {a[:, 2].std(ddof=1) if len(a) > 1 else 0:.0f}",
                 'rmse_mean': a[:, 0].mean()})
MAIN = pd.DataFrame(rows)
MAIN.to_csv(RESULTS / 'main_results.csv', index=False)
display(MAIN.drop(columns='rmse_mean'))
print(f"Mean fusion weight α (DL share): {META.alpha.mean():.2f} ± {META.alpha.std():.2f}")

### 5.2 Statistical comparisons (point 5) and falsification with the hybrid model

The *Falsification (Hybrid)* family repeats the key W11 controls with the final model: if the gain of the 18 BI comes from each engine's own history, the real data must beat the shuffled versions, and the shuffled versions must not beat sensors only.

Engine-level errors (mean absolute error over the 3 truncation points and 3 seeds) are compared with a one-sided Wilcoxon signed-rank test (is the first configuration better?), Holm-corrected within each family, plus a bootstrap 95% CI of ΔRMSE (2,000 resamples of engines).

In [ ]:
def engine_err(fset, model):
    g = PRED[PRED.fset == fset].assign(e=lambda x: x[model] - x.true_rul)
    return g.groupby('unit').agg(abs_err=('e', lambda e: np.mean(np.abs(e))), sq_err=('e', lambda e: np.mean(e ** 2)))

BOOT = np.random.default_rng(0).integers(0, len(UNITS), (2000, len(UNITS)))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for i, j in enumerate(o):
        run = max(run, (len(p) - i) * p[j]); adj[j] = min(run, 1)
    return adj

def compare(a, b):
    ea, eb = engine_err(*a), engine_err(*b)
    d = np.sqrt(ea.sq_err.mean()) - np.sqrt(eb.sq_err.mean())
    bt = np.sqrt(ea.sq_err.values[BOOT].mean(1)) - np.sqrt(eb.sq_err.values[BOOT].mean(1))
    diff = ea.abs_err.values - eb.abs_err.values
    p = stats.wilcoxon(diff, alternative='less').pvalue if np.any(diff != 0) else 1.0
    return {'comparison': f"{MODEL_NAME[a[1]]} [{a[0]}]  vs  {MODEL_NAME[b[1]]} [{b[0]}]",
            'dRMSE': d, 'CI_low': np.percentile(bt, 2.5), 'CI_high': np.percentile(bt, 97.5), 'p_raw': p}

FAMILIES = {
    'BI value (Hybrid)': [(('S+Op', 'hybrid'), ('S', 'hybrid')), (('S+Op+A', 'hybrid'), ('S', 'hybrid')),
                          (('S+Op+A', 'hybrid'), ('S+Op', 'hybrid'))],
    'BI value (ML / DL)': [(('S+Op+A', 'ml'), ('S', 'ml')), (('S+Op+A', 'dl'), ('S', 'dl'))],
    'Hybrid vs branches': [((f, 'hybrid'), (f, m)) for f in MAIN_FSETS for m in ('ml', 'dl')],
    'Fusion and ML learner': [((f, 'hybrid'), (f, 'hybrid_dense')) for f in MAIN_FSETS] +
                             [((f, 'ml'), (f, 'rf')) for f in MAIN_FSETS],
    'Falsification (Hybrid)': [(('S+Op+A', 'hybrid'), ('S+Op+A~', 'hybrid')),    # real vs shuffled asset history
                               (('S+Op+A', 'hybrid'), ('S+Op~+A~', 'hybrid')),   # real vs all BI shuffled
                               (('S+Op+A~', 'hybrid'), ('S', 'hybrid')),        # shuffled asset vs sensors
                               (('S+Op~+A~', 'hybrid'), ('S', 'hybrid'))],      # all shuffled vs sensors
}
rows = []
for fam, pairs in FAMILIES.items():
    res = [compare(a, b) for a, b in pairs]
    for r, ph in zip(res, holm([r['p_raw'] for r in res])):
        rows.append({'family': fam, **r, 'p_holm': ph})
STATS = pd.DataFrame(rows)
STATS['sig'] = pd.cut(STATS.p_holm, [-1, 0.001, 0.01, 0.05, 2], labels=['***', '**', '*', 'ns'])
STATS.to_csv(RESULTS / 'stats.csv', index=False)
display(STATS.round(3))

### 5.3 Branch complementarity (point 6)

Do the ML and DL branches make different errors? Low residual correlation and a balanced share of engines won by each branch justify the fusion.

In [ ]:
rows = []
for fset in MAIN_FSETS:
    g = PRED[PRED.fset == fset]
    e_ml, e_dl = g.ml - g.true_rul, g.dl - g.true_rul
    eng = g.assign(a_ml=np.abs(e_ml), a_dl=np.abs(e_dl)).groupby('unit')[['a_ml', 'a_dl']].mean()
    oracle = np.where(np.abs(e_ml) < np.abs(e_dl), g.ml, g.dl)
    rows.append({'Features': fset,
                 'Residual corr ρ(e_ML, e_DL)': stats.pearsonr(e_ml, e_dl)[0],
                 '% engines ML better': 100 * (eng.a_ml < eng.a_dl).mean(),
                 '% engines DL better': 100 * (eng.a_dl < eng.a_ml).mean(),
                 'RMSE ML': rmse(g.true_rul, g.ml), 'RMSE DL': rmse(g.true_rul, g.dl),
                 'RMSE Hybrid': rmse(g.true_rul, g.hybrid),
                 'RMSE oracle (best branch per point)': rmse(g.true_rul, oracle)})
COMPL = pd.DataFrame(rows).round(3)
COMPL.to_csv(RESULTS / 'complementarity.csv', index=False)
display(COMPL)

### 5.4 Subgroup performance (point 18)

Hybrid RMSE by life stage at the truncation point, by production priority at that moment, and by asset-history group, and by C-MAPSS subset (FD001 / FD003 engines). Sensors only vs the full 18-variable BI set.

In [ ]:
ctx = CUTS.merge(bi_op[['unit', 'cycle', 'production_priority', 'contract_penalty_active']],
                 left_on=['unit', 'cut_cycle'], right_on=['unit', 'cycle'], how='left').drop(columns='cycle')
ctx = ctx.merge(asset[['unit', 'supplier', 'last_overhaul_type']], on='unit')
ctx['life stage'] = pd.cut(ctx.true_rul, [-1, 30, 80, RUL_MAX], labels=['near failure (≤30)', 'mid-life (31–80)', 'early (>80)'])
ctx['production priority'] = ctx.production_priority.map({0: 'low', 1: 'medium', 2: 'high'})
ctx['supplier'] = ctx.supplier.map({0: 'OEM', 1: 'alternative'})
ctx['last overhaul'] = ctx.last_overhaul_type.map({0: 'light', 1: 'full'})
ctx['contract penalty'] = ctx.contract_penalty_active.map({0: 'inactive', 1: 'active'})
ctx['C-MAPSS subset'] = ctx.unit.map(train.groupby('unit').subset.first())

rows = []
for dim in ['C-MAPSS subset', 'life stage', 'production priority', 'supplier', 'last overhaul', 'contract penalty']:
    for lvl, sub in ctx.groupby(dim):
        r = {'dimension': dim, 'group': lvl, 'n points': len(sub) * len(SEEDS)}
        for fset in ['S', 'S+Op+A']:
            g = PRED[PRED.fset == fset].merge(sub[['unit', 'cut_id']], on=['unit', 'cut_id'])
            r[f'RMSE Hybrid [{fset}]'] = rmse(g.true_rul, g.hybrid)
        r['Δ (18 BI − sensors)'] = r['RMSE Hybrid [S+Op+A]'] - r['RMSE Hybrid [S]']
        rows.append(r)
SUB = pd.DataFrame(rows).round(2)
SUB.to_csv(RESULTS / 'subgroups.csv', index=False)
display(SUB)

### 5.5 Late vs early predictions (point 19)

A prediction above the true RUL is **late** (optimistic): maintenance would come after it should, which risks failure. A prediction below is **early** (conservative): it costs some useful life but is safe. The NASA score already penalises late errors more.

In [ ]:
rows = []
for fset, model in itertools.product(MAIN_FSETS, MODELS):
    g = PRED[PRED.fset == fset]; e = g[model] - g.true_rul
    for stage, mask in [('all', np.ones(len(g), bool))] + \
            [(s, (pd.cut(g.true_rul, [-1, 30, 80, RUL_MAX], labels=['near', 'mid', 'early']) == s).values)
             for s in ['near', 'mid', 'early']]:
        es = e[mask]
        rows.append({'Features': fset, 'Model': MODEL_NAME[model], 'stage': stage,
                     'P(late)': (es > 0).mean(), 'P(early)': (es < 0).mean(),
                     'median late error': es[es > 0].median(), 'median early error': -es[es < 0].median(),
                     'P(late > 20 cycles)': (es > 20).mean()})
LATE = pd.DataFrame(rows).round(3)
LATE.to_csv(RESULTS / 'late_vs_early.csv', index=False)
display(LATE[LATE.stage == 'all'])
display(LATE[(LATE.Model == 'Hybrid')])

### 5.6 Uncertainty: conformal intervals and failure risk (points 10 and 21)

Cross-conformal intervals from out-of-fold residuals: for each test fold, the residuals of the **other** folds (same seed) calibrate a 90% interval, separately by predicted life stage (Mondrian conformal, as in the bearing paper). The same residuals give the probability that the engine fails within a horizon *h*: P(RUL ≤ h) = share of calibration residuals for which ŷ + r ≤ h.

In [ ]:
ALPHA_CONF = 0.10
HORIZONS = [15, 30]
def stage_of(p): return pd.cut(p, [-np.inf, 30, 80, np.inf], labels=['near', 'mid', 'early'])

rows, risk_rows = [], []
for fset in MAIN_FSETS:
    for seed in SEEDS:
        g = PRED[(PRED.fset == fset) & (PRED.seed == seed)].copy()
        g['r'] = g.true_rul - g.hybrid; g['st'] = stage_of(g.hybrid)
        for k in range(N_FOLDS):
            te, cal = g[g.fold == k], g[g.fold != k]
            for st, sub in te.groupby('st'):
                rc = cal.loc[cal.st == st, 'r'].values
                if len(rc) < 10 or len(sub) == 0:
                    continue
                q = np.quantile(np.abs(rc), min(1, np.ceil((len(rc) + 1) * (1 - ALPHA_CONF)) / len(rc)))
                rows.append({'fset': fset, 'stage': st, 'n': len(sub),
                             'covered': int(np.sum(np.abs(sub.r) <= q)), 'width': 2 * q})
            for h in HORIZONS:
                rc_all = cal[['st', 'r']]
                for _, row in te.iterrows():
                    rc = rc_all.loc[rc_all.st == row.st, 'r'].values
                    risk_rows.append({'fset': fset, 'h': h, 'p_fail': float(np.mean(row.hybrid + rc <= h)),
                                      'failed': int(row.true_rul <= h)})
CONF = pd.DataFrame(rows).groupby(['fset', 'stage']).apply(
    lambda x: pd.Series({'coverage (target 0.90)': x.covered.sum() / x.n.sum(),
                         'mean width (cycles)': np.average(x.width, weights=x.n)})).round(3)
CONF.to_csv(RESULTS / 'conformal.csv')
display(CONF)

RISK = pd.DataFrame(risk_rows)
risk_sum = RISK.groupby(['fset', 'h']).apply(lambda x: pd.Series({
    'Brier score': np.mean((x.p_fail - x.failed) ** 2),
    'Brier (no-skill)': np.mean((x.failed.mean() - x.failed) ** 2),
    'mean risk | failed': x.loc[x.failed == 1, 'p_fail'].mean(),
    'mean risk | not failed': x.loc[x.failed == 0, 'p_fail'].mean()})).round(3)
risk_sum.to_csv(RESULTS / 'failure_risk.csv')
display(risk_sum)

### 5.7 Feature importance and explanation stability (points 16 and 15)

- **Permutation importance (point 16):** increase in RMSE of the ML branch when one variable is shuffled on the held-out points (10 repeats), averaged over folds and seeds.
- **Stability (point 15):** Spearman rank correlation of the importance vectors between every pair of runs (folds × seeds), and overlap of the top-10 variables. High values mean the explanations do not depend on which engines were used for training.

In [ ]:
g = IMP[IMP.fset == 'S+Op+A']
agg = g.groupby('feature')[['perm_ml', 'shap_ml', 'attn_dl']].mean()
agg['group'] = [GROUP_OF.get(f, 'other') for f in agg.index]
PERM_TOP = agg.sort_values('perm_ml', ascending=False).head(20).round(4)
display(PERM_TOP)
print("Share of total importance by group (18-BI model):")
display((agg.groupby('group')[['perm_ml', 'shap_ml', 'attn_dl']].sum() /
         agg[['perm_ml', 'shap_ml', 'attn_dl']].clip(lower=0).sum()).round(3))
agg.to_csv(RESULTS / 'importance_18bi.csv')

def stability(col):
    runs = [x.set_index('feature')[col].dropna() for _, x in g.groupby(['seed', 'fold'])]
    rhos, jac = [], []
    for a_, b_ in itertools.combinations(runs, 2):
        common = a_.index.intersection(b_.index)
        if len(common) > 3:
            rhos.append(stats.spearmanr(a_[common], b_[common])[0])
        ta, tb = set(a_.nlargest(10).index), set(b_.nlargest(10).index)
        jac.append(len(ta & tb) / len(ta | tb))
    return {'mean Spearman ρ': np.mean(rhos), 'min Spearman ρ': np.min(rhos), 'mean top-10 Jaccard': np.mean(jac)}
STAB = pd.DataFrame({m: stability(c) for m, c in
                     [('SHAP (ML)', 'shap_ml'), ('Permutation (ML)', 'perm_ml'), ('Feature attention (DL)', 'attn_dl')]}).T.round(3)
STAB.to_csv(RESULTS / 'explanation_stability.csv')
display(STAB)

fig, ax = plt.subplots(figsize=(9, 6))
top = agg.sort_values('perm_ml', ascending=True).tail(15)
ax.barh(top.index, top.perm_ml, color=top.group.map({'sensor': '#6baed6', 'operational BI': '#fd8d3c', 'asset BI': '#31a354'}))
ax.set_xlabel('Permutation importance (ΔRMSE, cycles)'); ax.set_title('Top 15 variables — ML branch, 18 BI variables')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=l) for l, c in
                   [('sensor', '#6baed6'), ('operational BI', '#fd8d3c'), ('asset BI', '#31a354')]])
plt.tight_layout(); plt.savefig(RESULTS / 'fig_permutation_importance.png', dpi=150); plt.show()

### 5.8 Computing time (point 22)

In [ ]:
TIME = META[META.fset.isin(MAIN_FSETS)].groupby('fset')[['t_select', 't_train_ml', 't_train_rf', 't_train_dl', 't_shap', 'dl_epochs']].mean()
TIME['inference ML (ms/engine)'] = META[META.fset.isin(MAIN_FSETS)].groupby('fset').t_inf_ml.mean() * 1000
TIME['inference DL (ms/engine)'] = META[META.fset.isin(MAIN_FSETS)].groupby('fset').t_inf_dl.mean() * 1000
TIME = TIME.rename(columns={'t_select': 'feature selection (s)', 't_train_ml': 'train XGBoost (s)', 't_train_rf': 'train RF (s)',
                            't_train_dl': 'train DL (s)', 't_shap': 'SHAP on 60 points (s)', 'dl_epochs': 'DL epochs'}).round(2)
TIME.to_csv(RESULTS / 'computing_time.csv')
print(f"Hardware: {'GPU ' + str(GPU) if GPU else 'CPU'} — times are per fold, mean over seeds")
display(TIME)

### 5.9 Summary figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
LABELS = {'S': 'Sensors', 'S+Op': '+ 13 op. BI', 'S+Op+A': '+ 18 BI', 'S+Op+A~': '18 BI,\nasset shuffled',
          'S+Op~+A~': '18 BI,\nall shuffled'}
x = np.arange(len(FSETS)); w = 0.16
for i, m in enumerate(MODELS):
    vals = [MAIN[(MAIN.Features == f) & (MAIN.Model == MODEL_NAME[m])].rmse_mean.iloc[0] for f in FSETS]
    ax.bar(x + (i - 2) * w, vals, w, label=MODEL_NAME[m])
ax.set_xticks(x); ax.set_xticklabels([LABELS[f] for f in FSETS], fontsize=8)
ax.set_ylabel('RMSE (cycles)'); ax.set_ylim(min(MAIN.rmse_mean) - 2, max(MAIN.rmse_mean) + 1)
ax.set_title('5-fold CV with truncation, mean of 3 seeds'); ax.legend()

ax = axes[1]
s = SUB[SUB.dimension == 'life stage']
ax.bar(np.arange(len(s)) - 0.2, s['RMSE Hybrid [S]'], 0.4, label='Sensors')
ax.bar(np.arange(len(s)) + 0.2, s['RMSE Hybrid [S+Op+A]'], 0.4, label='+ 18 BI')
ax.set_xticks(np.arange(len(s))); ax.set_xticklabels(s.group); ax.set_ylabel('Hybrid RMSE'); ax.legend()
ax.set_title('Where the BI helps: by life stage')
plt.tight_layout(); plt.savefig(RESULTS / 'fig_main_results.png', dpi=150); plt.show()

if SAVE_DIR is not None:
    for f in RESULTS.iterdir():
        if f.is_file():
            shutil.copy(f, SAVE_DIR / f.name)
    print(f"Results copied to {SAVE_DIR}")

## 6. Reading the results

- **Main table (5.1):** RQ1 (BI predictive value) compares the three main feature sets; RQ2 (hybrid complementarity) compares Hybrid with each branch, supported by 5.3.
- **Falsification with the final model (5.2):** the 18-BI gain is credible only if real data beat both shuffled controls and the shuffled controls do not beat sensors only.
- **Statistics (5.2):** report ΔRMSE with its CI and the Holm-corrected p-value, not only the RMSE difference.
- **Routing (4.1):** report the first run (asset history to both branches) as the evidence for routing static records to the ML branch only.
- **Subgroups (5.4):** the asset-history BI should help mostly in mid-life, consistent with W11.
- **Late errors (5.5) and failure risk (5.6):** feed directly into the decision layer (W09).
- **Reporting:** state that BI variables are synthetic and how they were generated, the value of ρ and why it was chosen, and cite C-MAPSS (Saxena et al., 2008). Results are 5-fold CV on the training engines with truncation, not the official test set.